[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/05_Entity_Extraction.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。

# 实体抽取（Entity Extraction）—— 全面指南

## 概览

本 notebook 提供一份**全面指南**，讲解如何用 Semantica 强大的命名实体识别（NER）模块从文本中抽取命名实体。你将学会使用多个抽取器、多种方法和高级特性来识别并分类文本中的实体。

**文档**：[API 参考](https://semantica.readthedocs.io/reference/semantic_extract/)

### 学习目标

完成本 notebook 后，你将能够：

- 使用 `NERExtractor` 和 `NamedEntityRecognizer` 抽取实体
- 理解不同的抽取方法（pattern、regex、ML、HuggingFace、LLM）
- 使用 `EntityClassifier` 对实体进行分类和分组
- 应用 `EntityConfidenceScorer` 评估抽取质量
- 用 `CustomEntityDetector` 创建自定义实体模式
- 配置抽取参数以获得最佳效果
- 高效处理多个文档
- 优雅地处理边界情况和错误

### 你将学到什么

| 组件 | 用途 | 适用场景 |
|-----------|---------|-------------|
| `NERExtractor` | 核心实体抽取 | 快速、简单的抽取 |
| `NamedEntityRecognizer` | 可配置的高级 NER | 需要精细控制 |
| `EntityClassifier` | 实体分类与分组 | 整理抽取出的实体 |
| `EntityConfidenceScorer` | 实体置信度打分 | 质量评估 |
| `CustomEntityDetector` | 领域特定实体 | 需要自定义模式 |

---

## 安装

从 PyPI 安装 Semantica：

```bash
pip install semantica
# 或安装全部可选依赖：
pip install semantica[all]
```

---

In [ ]:
!pip install semantica


## 第 1 步：用 NERExtractor 做基础实体抽取

我们从最简单的 `NERExtractor` 用起。这个类为从文本中抽取命名实体提供了直观的接口。

### 什么是 NERExtractor？

`NERExtractor` 是核心实体抽取类，它可以：
- 识别命名实体（人物、组织、地点、日期等）
- 返回包含文本、类型、位置和置信度的实体对象
- 支持多种抽取方法
- 默认配置合理，开箱即用

In [ ]:
from semantica.semantic_extract import NERExtractor

# Initialize the extractor
ner_extractor = NERExtractor()

# Sample text with various entity types
text = """
Apple Inc. is a technology company founded by Steve Jobs, Steve Wozniak, and Ronald Wayne 
in Cupertino, California on April 1, 1976. The company's current CEO is Tim Cook, who took 
over from Steve Jobs in August 2011. Apple is headquartered at One Apple Park Way in Cupertino.
"""

# Extract entities
entities = ner_extractor.extract(text)

print(f" Extracted {len(entities)} entities:\n")
print("-" * 80)

for i, entity in enumerate(entities, 1):
    # Handle both dict and object formats
    entity_text = entity.get('text', entity.get('entity', '')) if isinstance(entity, dict) else entity.text
    entity_type = entity.get('type', entity.get('label', 'Unknown')) if isinstance(entity, dict) else entity.label
    confidence = entity.get('confidence', 1.0) if isinstance(entity, dict) else getattr(entity, 'confidence', 1.0)
    
    print(f"{i:2d}. {entity_text:30s} | Type: {entity_type:12s} | Confidence: {confidence:.2f}")

print("-" * 80)

### 理解实体对象

每个抽取出的实体包含：

| 属性 | 描述 | 示例 |
|-----------|-------------|----------|
| `text` | 实体文本 | "Apple Inc." |
| `label/type` | 实体类别 | "ORG"（组织） |
| `start_char` | 起始位置 | 0 |
| `end_char` | 结束位置 | 10 |
| `confidence` | 抽取置信度（0-1） | 0.95 |
| `metadata` | 附加信息 | {"method": "ml"} |

### 常见实体类型

- **PERSON**：人物，包括虚构角色
- **ORG**：公司、机构、组织
- **GPE**：国家、城市、州（地缘政治实体）
- **LOC**：非 GPE 地点，如山脉、水体
- **DATE**：绝对或相对的日期或时期
- **TIME**：小于一天的时间
- **MONEY**：货币金额
- **PERCENT**：百分比值

## 第 2 步：在上下文中可视化实体

我们来做一个简单的可视化，看看实体在原文中被标出的效果。

In [ ]:
def highlight_entities(text, entities):
    """
    Create a simple text visualization with entity markers.
    """
    # Group entities by type
    entity_types = {}
    for entity in entities:
        entity_text = entity.get('text', entity.get('entity', '')) if isinstance(entity, dict) else entity.text
        entity_type = entity.get('type', entity.get('label', 'Unknown')) if isinstance(entity, dict) else entity.label
        
        if entity_type not in entity_types:
            entity_types[entity_type] = []
        entity_types[entity_type].append(entity_text)
    
    print("\n Entity Visualization:\n")
    print("=" * 80)
    
    for entity_type, entity_list in sorted(entity_types.items()):
        unique_entities = list(set(entity_list))
        print(f"\n{entity_type}:")
        for ent in unique_entities:
            print(f"  • {ent}")
    
    print("\n" + "=" * 80)

# Visualize the extracted entities
highlight_entities(text, entities)

## 第 3 步：不同的抽取方法

Semantica 支持多种抽取方法，各有所长：

### 方法对比

| 方法 | 速度 | 准确率 | 适用场景 | 依赖 |
|--------|-------|----------|----------|----------|
| **pattern** |  | ⭐⭐ | 简单、可预测的模式 | 无 |
| **regex** |  | ⭐⭐⭐ | 自定义模式、ID、编码 | 正则知识 |
| **ml**（spaCy） |  | ⭐⭐⭐⭐ | 通用文本、多语言 | spaCy 模型 |
| **huggingface** |  | ⭐⭐⭐⭐⭐ | 领域特定、微调模型 | HF 模型 |
| **llm** |  | ⭐⭐⭐⭐⭐ | 复杂、自定义类型 | API key |

下面试试不同的方法：

In [ ]:
from semantica.semantic_extract import NERExtractor

sample_text = "Apple Inc. was founded by Steve Jobs in Cupertino, California in 1976."

print(" Comparing Extraction Methods:\n")
print("=" * 80)

# Try different methods
methods_to_try = ["pattern", "regex", "ml"]

for method_name in methods_to_try:
    try:
        print(f"\n Method: {method_name.upper()}")
        print("-" * 40)
        
        extractor = NERExtractor(method=method_name)
        entities = extractor.extract(sample_text)
        
        print(f"Found {len(entities)} entities:")
        for entity in entities[:5]:  # Show first 5
            entity_text = entity.get('text', entity.get('entity', '')) if isinstance(entity, dict) else entity.text
            entity_type = entity.get('type', entity.get('label', 'Unknown')) if isinstance(entity, dict) else entity.label
            print(f"  • {entity_text} ({entity_type})")
            
    except Exception as e:
        print(f"  ️  Method '{method_name}' not available: {str(e)[:50]}")

print("\n" + "=" * 80)

## 第 4 步：用 NamedEntityRecognizer 做高级实体识别

`NamedEntityRecognizer` 通过配置参数对抽取过程提供更多控制。

### 关键参数：

- **`methods`**：要使用的抽取方法列表（如 `["spacy", "rule-based"]`）
- **`confidence_threshold`**：最低置信度分数（0.0-1.0，默认 0.5）
- **`merge_overlapping`**：是否合并重叠实体（默认 True）
- **`include_standard_types`**：包含标准实体类型（PERSON、ORG、LOC 等）

In [ ]:
from semantica.semantic_extract import NamedEntityRecognizer

# Create recognizer with custom configuration
ner = NamedEntityRecognizer(
    methods=["spacy"],  # Use spaCy for ML-based extraction
    confidence_threshold=0.7,  # Only keep high-confidence entities
    merge_overlapping=True,  # Merge overlapping entity mentions
    include_standard_types=True  # Include standard entity types
)

# Sample texts for batch processing
texts = [
    "Tim Cook is the CEO of Apple Inc., based in Cupertino.",
    "Microsoft Corporation, founded by Bill Gates, is headquartered in Redmond, Washington.",
    "Amazon was founded by Jeff Bezos in Seattle in 1994.",
    "Google was started by Larry Page and Sergey Brin at Stanford University."
]

print(" Advanced Entity Recognition Results:\n")
print("=" * 80)

all_entities = []
for i, text in enumerate(texts, 1):
    entities = ner.extract_entities(text)
    all_entities.extend(entities)
    
    print(f"\n Text {i}: {text[:60]}...")
    print(f"   Found {len(entities)} entities:")
    
    for entity in entities:
        entity_text = entity.get('text', entity.get('entity', '')) if isinstance(entity, dict) else entity.text
        entity_type = entity.get('type', entity.get('label', 'Unknown')) if isinstance(entity, dict) else entity.label
        confidence = entity.get('confidence', 1.0) if isinstance(entity, dict) else getattr(entity, 'confidence', 1.0)
        print(f"     • {entity_text:25s} | {entity_type:10s} | Confidence: {confidence:.2f}")

print(f"\n Total entities extracted: {len(all_entities)}")
print("=" * 80)

## 第 5 步：实体分类

使用 `EntityClassifier` 按类型对实体分类和分组，并对相似实体消歧。

### 什么是 EntityClassifier？

`EntityClassifier` 可以帮你：
- 按类型**分类实体**（归一 "ORG" 与 "ORGANIZATION" 这类变体）
- 按类别**分组实体**，便于分析
- 存在多个候选时对实体**消歧**
- 跨抽取方法**标准化实体类型**

In [ ]:
from semantica.semantic_extract import EntityClassifier

# Initialize classifier
classifier = EntityClassifier()

# Classify the entities we extracted earlier
classified = classifier.classify_entities(all_entities)

print("️  Entity Classification Results:\n")
print("=" * 80)

for entity_type, entity_list in sorted(classified.items()):
    print(f"\n{entity_type} ({len(entity_list)} entities):")
    print("-" * 40)
    
    # Get unique entity texts
    unique_entities = set()
    for entity in entity_list:
        entity_text = entity.get('text', entity.get('entity', '')) if isinstance(entity, dict) else entity.text
        unique_entities.add(entity_text)
    
    for entity_text in sorted(unique_entities):
        print(f"  • {entity_text}")

print("\n" + "=" * 80)

## 第 6 步：置信度打分

使用 `EntityConfidenceScorer` 评估并改进抽取实体的置信度分数。

### 为什么需要置信度打分？

置信度分数可以帮你：
- **过滤低质量抽取**（例如只保留置信度 > 0.8 的实体）
- **排定优先级**，决定哪些实体需要人工审核或验证
- **评估抽取质量**，比较不同方法或不同文本
- **做出明智决策**，决定使用哪些实体

In [ ]:
from semantica.semantic_extract import EntityConfidenceScorer

# Initialize confidence scorer
scorer = EntityConfidenceScorer()

# Score the entities
scored_entities = scorer.score_entities(all_entities)

print(" Entity Confidence Scoring:\n")
print("=" * 80)

# Group by confidence levels
high_confidence = []
medium_confidence = []
low_confidence = []

for entity in scored_entities:
    confidence = entity.get('confidence', 1.0) if isinstance(entity, dict) else getattr(entity, 'confidence', 1.0)
    
    if confidence >= 0.8:
        high_confidence.append(entity)
    elif confidence >= 0.5:
        medium_confidence.append(entity)
    else:
        low_confidence.append(entity)

print(f" High Confidence (≥0.8): {len(high_confidence)} entities")
print(f"️  Medium Confidence (0.5-0.8): {len(medium_confidence)} entities")
print(f" Low Confidence (<0.5): {len(low_confidence)} entities")

print("\n Confidence Distribution:")
print("-" * 40)

# Show some examples from each category
if high_confidence:
    print("\nHigh Confidence Examples:")
    for entity in high_confidence[:3]:
        entity_text = entity.get('text', entity.get('entity', '')) if isinstance(entity, dict) else entity.text
        entity_type = entity.get('type', entity.get('label', 'Unknown')) if isinstance(entity, dict) else entity.label
        confidence = entity.get('confidence', 1.0) if isinstance(entity, dict) else getattr(entity, 'confidence', 1.0)
        print(f"  • {entity_text} ({entity_type}) - {confidence:.2f}")

print("\n" + "=" * 80)

## 第 7 步：自定义实体检测

使用 `CustomEntityDetector` 定义领域特定的实体模式。

### 什么时候用自定义模式？

自定义模式适合：
- **领域特定实体**（如产品编码、发票号）
- **结构化标识符**（如电子邮箱、电话号码）
- **行业术语**（如医疗编码、法律引注）
- 标准 NER 无法识别的**自定义格式**

In [ ]:
from semantica.semantic_extract import CustomEntityDetector
import re

# Define custom patterns
custom_patterns = {
    "EMAIL": r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
    "PHONE": r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b',
    "PRODUCT_CODE": r'\b[A-Z]{2,3}-\d{4,6}\b',
    "URL": r'https?://[^\s]+'
}

# Initialize custom detector
custom_detector = CustomEntityDetector(patterns=custom_patterns)

# Sample text with custom entities
custom_text = """
For support, contact support@apple.com or call 1-800-692-7753.
Order product SKU-12345 from https://store.apple.com.
Technical inquiries: tech@apple.com or visit our website.
"""

print(" Custom Entity Detection:\n")
print("=" * 80)

for entity_type in custom_patterns.keys():
    entities = custom_detector.detect_custom_entities(custom_text, entity_type)
    
    if entities:
        print(f"\n{entity_type}:")
        for entity in entities:
            entity_text = entity.get('text', entity.get('entity', '')) if isinstance(entity, dict) else entity.text
            print(f"  • {entity_text}")

print("\n" + "=" * 80)

## 第 8 步：批处理

使用批处理能力高效处理多个文档。

### 批处理的好处：

- **性能**：一次调用处理多个文档
- **一致性**：同一配置应用到所有文档
- **效率**：减少初始化开销
- **可扩展**：应对大规模文档集合

In [ ]:
# Sample document collection
documents = [
    "Apple Inc. released the iPhone 15 in September 2023.",
    "Microsoft announced Azure AI updates at Build 2023 in Seattle.",
    "Google's Sundar Pichai spoke at I/O 2023 in Mountain View, California.",
    "Tesla's Elon Musk unveiled the Cybertruck in Austin, Texas.",
    "Amazon Web Services launched new features in Northern Virginia."
]

print(" Batch Processing Results:\n")
print("=" * 80)

# Process all documents
batch_results = ner.process_batch(documents)

# Analyze results
total_entities = 0
entity_type_counts = {}

for i, (doc, entities) in enumerate(zip(documents, batch_results), 1):
    total_entities += len(entities)
    
    print(f"\n Document {i}:")
    print(f"   Text: {doc[:50]}...")
    print(f"   Entities: {len(entities)}")
    
    for entity in entities:
        entity_type = entity.get('type', entity.get('label', 'Unknown')) if isinstance(entity, dict) else entity.label
        entity_type_counts[entity_type] = entity_type_counts.get(entity_type, 0) + 1

print(f"\n Batch Processing Summary:")
print("-" * 40)
print(f"Documents processed: {len(documents)}")
print(f"Total entities: {total_entities}")
print(f"Average per document: {total_entities/len(documents):.1f}")

print("\n Entity Type Distribution:")
for entity_type, count in sorted(entity_type_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {entity_type}: {count}")

print("\n" + "=" * 80)

## 第 9 步：最佳实践与技巧

### 选择合适的方法

1. 通用文本**从 ML（spaCy）开始**
2. 结构化数据（ID、编码）**用 pattern/regex**
3. 领域特定需求**试试 HuggingFace**
4. 复杂、自定义实体类型**考虑 LLM**

### 优化性能

- **设置合适的置信度阈值**（生产环境 0.7-0.8）
- 多文档**使用批处理**
- **开启 merge_overlapping** 减少重复
- **复用抽取器**，不要反复重建

### 常见的坑

- **不要**用很低的置信度阈值（< 0.5）
- **不要**在循环里一个一个处理文档
- **不要**忽略实体元数据（里面有有用信息）
- **不要**忘记处理抽取错误

### 各类的适用场景

| 使用场景 | 推荐类 |
|----------|-------------------|
| 快速抽取 | `NERExtractor` |
| 精细控制 | `NamedEntityRecognizer` |
| 实体分组 | `EntityClassifier` |
| 质量评估 | `EntityConfidenceScorer` |
| 领域特定 | `CustomEntityDetector` |

## 小结

### 你学到了什么

在本 notebook 中，你学会了：

用 `NERExtractor` 和 `NamedEntityRecognizer` **抽取实体**  
**对比不同的抽取方法**（pattern、regex、ML、HuggingFace、LLM）  
用 `EntityClassifier` **分类和分组实体**  
用 `EntityConfidenceScorer` **给实体置信度打分**  
用 `CustomEntityDetector` **创建自定义模式**  
**批量处理文档**提升效率  
**应用最佳实践**用于生产环境  

### 核心要点

1. **多种方法可选**：按需权衡（速度 vs 准确率）
2. **配置很重要**：调好参数才有最佳效果
3. **置信度是关键**：用阈值过滤低质量抽取
4. **自定义模式有效**：适合领域特定实体
5. **批处理可扩展**：高效处理多个文档

### 下一步

**下一篇**：[06_Relation_Extraction.ipynb](./06_Relation_Extraction.ipynb)  
学习如何抽取你已识别的实体之间的关系！

**延伸阅读**：
- [Semantic Extract API 参考](https://semantica.readthedocs.io/reference/semantic_extract/)
- [高级抽取技术](../advanced/01_Advanced_Extraction.ipynb)
- [构建知识图谱](./07_Building_Knowledge_Graphs.ipynb)

---

**有疑问或问题？** 欢迎访问我们的 [GitHub 仓库](https://github.com/semantica-agi/semantica)或[文档](https://semantica.readthedocs.io)。